# 🔲 Cellular Automata: Emergent Pathfinding from Simple Grid Agents

Here the environment itself is the agent system. Every cell of a grid is a mini-agent with one local rule:
*"my distance to the target is 1 + the smallest distance among my neighbours"*. There is no controller and no
global pathfinding algorithm. Applied to every cell at once, tick after tick, the rule spreads a distance
**wave** from the target around obstacles. A trivially simple carrier then walks downhill on that wave.

The example is a **warehouse**: items on shelves must reach a packing station.

## Learning Objectives
In this notebook, you will learn:
1. **Cell agents** - A local, synchronous update rule on a grid of simple agents
2. **Wave propagation** - Tick until the grid stabilizes, and count the ticks
3. **Gradient descent readout** - Follow decreasing values from an item to the target
4. **Verifying emergence** - Check the emergent path length against a global BFS
5. **Adaptability** - Add an obstacle, re-propagate and watch the path reroute

## Where this fits
Every earlier notebook in this module had a few capable LLM agents and some form of coordinator. This one takes
the opposite extreme: many trivial agents and **no coordinator**. It is the last notebook of the module;
module 09 returns to agents running in parallel, starting with how LangGraph schedules them.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`, only for the optional LLM summary in the last step

| Property | Value |
|---|---|
| Origin | Cellular automata (von Neumann 1966; Conway's *Game of Life*, 1970) |


### Definition
A **Grid-Based Agent System** is an architecture where a large number of simple agents (or "cells") are arranged in a spatial grid. Each agent has a state and updates it synchronously based on a rule set that only considers the states of its immediate neighbors. Complex, high-level patterns and problem-solving capabilities emerge from these local interactions.

### High-level Workflow

1.  **Grid Initialization:** A grid of cell-agents is created, each initialized with a type (e.g., obstacle, empty) and a state (e.g., a value).
2.  **Set Boundary Conditions:** One or more cells are given a special state to start a computation (e.g., a "target" cell's value is set to 0).
3.  **Synchronous Tick:** The system "ticks" forward. In each tick, every cell simultaneously calculates its next state based on the current state of its neighbors.
4.  **Emergence:** As the system ticks, information propagates across the grid like a wave. This can create gradients, paths, and other complex structures.
5.  **State Stabilization:** The system runs until the grid state stabilizes (no more changes occur), indicating that the computation is complete.
6.  **Readout:** The solution to the problem is then read directly from the final state of the grid (e.g., by following a computed gradient).

### When to Use / Applications
*   **Spatial Reasoning & Logistics:** Optimal pathfinding in dynamic environments (like our warehouse example).
*   **Complex System Simulation:** Modeling phenomena with emergent behavior like forest fires, disease spread, or urban growth.
*   **Parallel Computation:** Certain algorithms can be mapped to a cellular automata model for execution on highly parallel hardware (like GPUs).

### Strengths & Weaknesses
*   **Strengths:**
    *   **High Parallelism:** The logic is inherently parallel, making it extremely fast on appropriate hardware.
    *   **Adaptability:** The system can dynamically react to changes in the environment (e.g., a new obstacle) by simply re-propagating its waves.
    *   **Emergent Complexity:** Can solve very complex problems with surprisingly simple rules.
*   **Weaknesses:**
    *   **Design Complexity:** Designing the local rules to produce the desired global behavior can be challenging and unintuitive.
    *   **Less Introspective:** It's harder to ask a single cell "why" it has a certain state; the reasoning is distributed across the entire system.

---
## 📦 Step 1: Environment Setup

The setup cell loads the project-root `.env` (only the optional last step calls an LLM) and imports numpy and rich.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from collections import deque
from typing import Dict, List, Optional, Tuple

import numpy as np
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

console = Console()
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY (optional last step only) from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

---
## 🏗️ Part 1: Building the Cellular Automata Environment

This is the most critical phase. We will define the two core classes for our simulation:
1.  `CellAgent`: Represents a single cell in the grid. It contains its type, its state (a pathfinding value), and the local rule for updating that state.
2.  `WarehouseGrid`: The container for the entire system. It will manage the 2D array of `CellAgent`s, orchestrate the synchronous `tick` updates, and handle visualization.

In [ ]:
# ============================================================================
# CELL AGENT AND WAREHOUSE GRID
# ============================================================================
class CellAgent:
    """A single agent in our grid. Its only job is to update its value based on neighbors."""
    def __init__(self, cell_type: str, item: Optional[str] = None):
        self.type = cell_type # 'EMPTY', 'OBSTACLE', 'SHELF', 'PACKING_STATION'
        self.item = item
        self.pathfinding_value = float('inf')

    def update_value(self, neighbors: List['CellAgent']):
        """The core local rule: my new value is 1 + the minimum value of my non-obstacle neighbors."""
        if self.type == 'OBSTACLE':
            return float('inf')

        min_neighbor_value = float('inf')
        for neighbor in neighbors:
            if neighbor.pathfinding_value < min_neighbor_value:
                min_neighbor_value = neighbor.pathfinding_value

        # The +1 represents the cost of moving from a neighbor to this cell
        return min(self.pathfinding_value, min_neighbor_value + 1)

class WarehouseGrid:
    """Manages the entire grid of CellAgents and the simulation loop."""
    def __init__(self, layout: List[str]):
        self.height = len(layout)
        self.width = len(layout[0])
        self.grid = self._create_grid_from_layout(layout)
        self.item_locations = self._get_item_locations()

    def _create_grid_from_layout(self, layout):
        grid = np.empty((self.height, self.width), dtype=object)
        for r, row_str in enumerate(layout):
            for c, char in enumerate(row_str):
                if char == ' ':
                    grid[r, c] = CellAgent('EMPTY')
                elif char == '#':
                    grid[r, c] = CellAgent('OBSTACLE')
                elif char == 'P':
                    grid[r, c] = CellAgent('PACKING_STATION')
                else: # It's an item
                    grid[r, c] = CellAgent('SHELF', item=char)
        return grid

    def _get_item_locations(self) -> Dict[str, Tuple[int, int]]:
        locations = {}
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r, c].type == 'SHELF':
                    locations[self.grid[r, c].item] = (r, c)
                if self.grid[r, c].type == 'PACKING_STATION':
                    locations['P'] = (r, c)
        return locations

    def neighbor_positions(self, r: int, c: int) -> List[Tuple[int, int]]:
        """In-bounds 4-neighbourhood (E, W, S, N) of a cell."""
        return [
            (r + dr, c + dc)
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]
            if 0 <= r + dr < self.height and 0 <= c + dc < self.width
        ]

    def get_neighbors(self, r: int, c: int) -> List[CellAgent]:
        return [self.grid[pos] for pos in self.neighbor_positions(r, c)]

    def tick(self) -> bool:
        """Performs one synchronous update of all cells. Returns True if any value changed."""
        changed = False
        # First, calculate all new values based on the current state
        new_values = np.empty((self.height, self.width))
        for r in range(self.height):
            for c in range(self.width):
                neighbors = self.get_neighbors(r, c)
                new_values[r, c] = self.grid[r, c].update_value(neighbors)

        # Then, apply all the new values
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r, c].pathfinding_value != new_values[r, c]:
                    self.grid[r, c].pathfinding_value = new_values[r, c]
                    changed = True
        return changed

    def visualize(self, show_values: bool = False, title: str = "Warehouse Grid"):
        """Displays the grid state using Rich."""
        table = Table(title=title, show_header=False, show_edge=True, padding=0)
        for _ in range(self.width):
            table.add_column(justify="center")

        for r in range(self.height):
            row_renderables = []
            for c in range(self.width):
                cell = self.grid[r, c]
                val = cell.pathfinding_value
                display_char = ''
                if cell.type == 'EMPTY': display_char = '[grey70]·[/grey70]'
                elif cell.type == 'OBSTACLE': display_char = '[red]█[/red]'
                elif cell.type == 'PACKING_STATION': display_char = '[bold green]P[/bold green]'
                elif cell.type == 'SHELF': display_char = f'[bold blue]{cell.item}[/bold blue]'

                if show_values and val != float('inf'):
                    # Color code the path values
                    color = int(255 - (val * 5) % 255)
                    row_renderables.append(f"[rgb({color},{color},{color}) on rgb(30,30,60)]{int(val):^3}[/]")
                else:
                    row_renderables.append(f" {display_char} ")
            table.add_row(*row_renderables)
        console.print(table)

print("Cellular Automata environment defined successfully.")

---
## 🌊 Part 2: Implementing the Emergent Behaviors

The grid itself is just a framework. We need to implement the high-level logic that uses the cellular automata to solve our problem. This involves two key emergent behaviors:

1.  **Path Wave Propagation:** A function that sets a target and lets the grid `tick` until a complete pathfinding gradient has formed across the entire warehouse floor.
2.  **Gradient Descent Traversal:** A function that simulates a "carrier" agent starting at an item's shelf and simply following the path of steepest descent (lowest `pathfinding_value`) until it reaches the target.

In [ ]:
# ============================================================================
# WAVE PROPAGATION AND GRADIENT TRAVERSAL
# ============================================================================
MAX_TICKS = 500  # loop guard; a grid of H x W cells stabilizes in at most H * W ticks


def propagate_path_wave(grid: WarehouseGrid, target_pos: Tuple[int, int]) -> int:
    """Reset every cell, seed the target with 0 and tick until nothing changes. Returns the tick count."""
    for r in range(grid.height):
        for c in range(grid.width):
            grid.grid[r, c].pathfinding_value = float("inf")
    grid.grid[target_pos].pathfinding_value = 0

    for tick in range(1, MAX_TICKS + 1):
        if not grid.tick():
            return tick
    raise RuntimeError(f"Grid did not stabilize within {MAX_TICKS} ticks")


def trace_and_move_item(grid: WarehouseGrid, start_pos: Tuple[int, int]) -> List[Tuple[int, int]]:
    """Follow strictly decreasing values from start_pos down to the target (value 0)."""
    path = [start_pos]
    r, c = start_pos
    while grid.grid[r, c].pathfinding_value > 0:
        best = min(grid.neighbor_positions(r, c), key=lambda pos: grid.grid[pos].pathfinding_value)
        if grid.grid[best].pathfinding_value >= grid.grid[r, c].pathfinding_value:
            console.print("[red]Path tracing got stuck: no downhill neighbour.[/red]")
            break
        path.append(best)
        r, c = best
    return path


print("✅ propagate_path_wave() and trace_and_move_item() defined")

---
## 🏭 Part 3: Fulfilling an Order

`fulfill_order` composes the two behaviours: for each item, propagate a wave from the packing station, then trace
the item's path down it. The value grid after the first wave shows the distance field the cells computed. Every
number is the length of the shortest route from that cell to `P`.

In [ ]:
# ============================================================================
# ORDER FULFILLMENT
# ============================================================================
def fulfill_order(layout: List[str], order: List[str], show_values: bool = True) -> Dict[str, List[Tuple[int, int]]]:
    grid = WarehouseGrid(layout)
    grid.visualize(title="Initial warehouse")
    packing_station_pos = grid.item_locations["P"]
    paths = {}

    for i, item_id in enumerate(order, start=1):
        item_pos = grid.item_locations.get(item_id)
        if not item_pos:
            console.print(Panel(f"[red]Item '{item_id}' not found.[/red]", title=f"Step {i}"))
            continue
        ticks = propagate_path_wave(grid, packing_station_pos)
        if show_values and i == 1:
            grid.visualize(show_values=True, title=f"Distance wave from P (stable after {ticks} ticks)")
        path = trace_and_move_item(grid, item_pos)
        paths[item_id] = path
        grid.grid[item_pos].type, grid.grid[item_pos].item = "EMPTY", None
        console.print(Panel(
            f"🌊 Wave stabilized after {ticks} ticks\n"
            f"🚚 Path ({len(path) - 1} moves): {' -> '.join(map(str, path))}",
            title=f"Step {i}: item '{item_id}'", border_style="blue",
        ))
    return paths


warehouse_layout = [
    "#######",
    "# D   #",
    "# ### #",
    "#A#C# #",
    "# # #B#",
    "#  P  #",
    "#######",
]
order_to_fulfill = ["A", "B"]
paths = fulfill_order(warehouse_layout, order_to_fulfill)

---
## ✅ Part 4: Is the Emergent Path Really the Shortest?

The introduction claims the local rule computes shortest paths. Check it against an independent, global
breadth-first search over the same grid (walls block movement; all other cells, including shelves, are
passable, matching the cell rule).

In [ ]:
# ============================================================================
# VERIFY AGAINST A GLOBAL BFS
# ============================================================================
def bfs_distance(layout: List[str], start: Tuple[int, int], goal: Tuple[int, int]) -> Optional[int]:
    queue, seen = deque([(start, 0)]), {start}
    while queue:
        (r, c), d = queue.popleft()
        if (r, c) == goal:
            return d
        for nr, nc in [(r, c + 1), (r, c - 1), (r + 1, c), (r - 1, c)]:
            if 0 <= nr < len(layout) and 0 <= nc < len(layout[0]) and layout[nr][nc] != "#" and (nr, nc) not in seen:
                seen.add((nr, nc))
                queue.append(((nr, nc), d + 1))
    return None


reference = WarehouseGrid(warehouse_layout).item_locations
for item_id, path in paths.items():
    bfs = bfs_distance(warehouse_layout, reference[item_id], reference["P"])
    moves = len(path) - 1
    print(f"Item {item_id}: cellular-automaton path {moves} moves | BFS shortest {bfs} | {'✅ equal' if moves == bfs else '❌ differs'}")

---
## 🧱 Part 5: Adaptability - Adding an Obstacle

Wall off the cell directly left of `P`, re-run the same wave, and the path for item `A` reroutes on its own. No
pathfinding code changes; only the environment does.

In [ ]:
# ============================================================================
# ADD AN OBSTACLE AND RE-PROPAGATE
# ============================================================================
blocked_layout = list(warehouse_layout)
blocked_layout[5] = "# #P  #"  # new wall at (5, 2), right next to P
print("\n".join(blocked_layout))

before = paths["A"]
after = fulfill_order(blocked_layout, ["A"], show_values=False)["A"]
print(f"\nItem A before: {len(before) - 1} moves via {before}")
print(f"Item A after:  {len(after) - 1} moves via {after}")
bfs = bfs_distance(blocked_layout, WarehouseGrid(blocked_layout).item_locations["A"], WarehouseGrid(blocked_layout).item_locations["P"])
print(f"BFS on the new layout: {bfs} moves | {'✅ equal' if len(after) - 1 == bfs else '❌ differs'}")

---
## 🤖 Part 6 (Optional): An LLM Summary of the Run

The grid needs no LLM. As an optional last step, a model turns the *measured* results (path lengths and BFS
checks) into a short logistics note. Skip this cell if you have no `OPENAI_API_KEY`.

In [ ]:
# ============================================================================
# LLM INITIALIZATION: optional: summarises the measured run
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# OPTIONAL LLM SUMMARY OF THE MEASURED RESULTS
# ============================================================================
facts = "\n".join(
    f"- Item {item_id}: moved from {path[0]} to the packing station in {len(path) - 1} moves"
    for item_id, path in paths.items()
)
facts += f"\n- After a new wall was added, item A's route changed from {len(before) - 1} to {len(after) - 1} moves"
summary = llm.invoke(
    "You are a logistics manager. In 3-4 sentences, summarise this order-fulfillment run for a colleague.\n\n" + facts
).content
console.print(Markdown(summary))

### Analysis of the Results
1. **No central planner:** no A* or BFS was used to *find* a path. The path is read off a distance field the cells
   computed together, one local update at a time.
2. **Information as a wave:** the value grid shows distance spreading from `P`, flowing around the walls. The
   grid computed the shortest distance from *every* open cell at once.
3. **Verified, not assumed:** Part 4 compares the emergent path with a global BFS; equal lengths confirm the local
   rule produced shortest paths.
4. **Adaptability:** in Part 5 a new wall changed the route with no change to the pathfinding code.

---
## 📝 Summary

- **Cell agents** follow one local, synchronous rule; global behaviour (a distance field) **emerges** from
  repeated ticks.
- **Readout** is trivial: a carrier walks strictly downhill from its item to the target.
- **Checks**: a `MAX_TICKS` guard stops non-converging runs, and a global BFS confirms the emergent paths are
  shortest.
- **Adaptability** comes free: change the environment, re-propagate.

### Next Steps
- Module 09, `09_async_and_parallel_multi_agent/notebooks/01_parallel_agents.ipynb`: how LangGraph runs nodes in
  parallel (supersteps, joins, reducers and `Send`), then async agents and streaming